<!-- NINO26-CABECALHO v1 -->
# 7A — Ciclo ENSO com redes neurais ConvLSTM

**Projeto NINO-BRASIL — Oceanografia Física UFPE — Thiago Vilar**  
**Código da fase/letra:** `7A`  ·  **Hipótese:** HIP4 (ConvLSTM do ciclo)

## Descritivo (por que este notebook existe)
Testa se campos espaciais do Pacífico (ConvLSTM) acrescentam skill sobre índices escalares e ML tabular — ENSO é propagação espaço-temporal, não só uma série do Niño 3.4.

## Pergunta
Campos espaciais do Pacífico acrescentam skill sobre índices escalares e ML tabular na caracterização/previsão do ciclo?

## Desafio (hipótese a testar)
Amostra observacional pequena (23 eventos) e vazamento na fronteira de sequências; exige embargo temporal e pré-treino em simulações.

## Metodologia (com referências)
ConvLSTM sobre sequências (tempo, lat, lon, canais) do Pacífico; normalização no treino, embargo, métricas por lead; pré-treino CMIP/NMME e fine-tuning observacional (Shi et al., 2015; Ham et al., 2019; Chattopadhyay et al., 2020; Taylor & Feng, 2022; Mir et al., 2024).

## Contrato de saídas — código predecessor único
Cada figura nasce do **mesmo** `registrar_figura(...)` que congela sua numeric-table sob o **mesmo código**, reescrevendo por **sobreposição** a cada execução:

```python
from nino_brasil.viz import registrar_figura
registrar_figura(fig, "Fig_7A01", fase=7, bloco="A",
                 titulo=..., descricao=..., hipotese="HIP4",
                 notebook="notebooks/fase7/7_ciclo_convlstm.ipynb",
                 fontes={"<tabela>": df})   # -> figures/fase7/<codigo>.png + numeric-tables/fase7/<codigo>/
```

| Código | Figura (`figures/fase7/<código>.png`) | Numeric-table (`numeric-tables/fase7/<código>/`) | Descrição |
|---|---|---|---|
| `Fig_7A01` | `Fig_7A01.png` | `Fig_7A01/` | skill por lead time (a treinar cientificamente) |
| `Fig_7A02` | `Fig_7A02.png` | `Fig_7A02/` | importância por oclusão de canais (a treinar) |

> Padrão em `docs/PADRAO_NOTEBOOKS.md`; validação por `python scripts/validar_figuras.py --strict`.

In [1]:
import sys, pathlib
ROOT=pathlib.Path.cwd()
while not (ROOT/'src').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src')); sys.path.insert(0,str(ROOT))
import pandas as pd; from IPython.display import Image, display
STATS=ROOT/'data/processed/parquet/statistics'


In [2]:
import nino_brasil.models.phase7_convlstm as p7
import numpy as np
# Preparacao de dados (sem TF): cubo sintetico p/ demonstrar a API de sequencias.
cube=np.random.rand(120,8,24,3).astype('float32'); fase=np.random.randint(0,4,120)
seq,y=p7.make_sequences(cube, fase, seq_len=24)
print('sequencias:', seq.shape, '| alvo:', y.shape)
tr,va=p7.chronological_split(len(seq)); print('treino/val cronologico:', len(tr), len(va))


sequencias: (97, 24, 8, 24, 3) | alvo: (97,)
treino/val cronologico: 78 19


## Construcao da rede (requer TensorFlow)
```python
model = p7.build_convlstm_classifier(input_shape=seq.shape[1:], n_classes=4)
model.fit(seq[tr], y[tr], validation_data=(seq[va], y[va]), epochs=30)
imp = p7.channel_occlusion_importance(model, seq[va])  # XAI espacial
```
Fonte real do cubo: `data/processed/zarr/regridded/` via `p7.load_pacific_cube(...)`.
So avanca se vencer climatologia, persistencia, Fase 3 e Fase 5.


<!-- NINO26-REFERENCIAS v1 -->
## Referências Bibliográficas

1. Shi, X., et al. (2015). Convolutional LSTM Network. *NeurIPS 28*. https://arxiv.org/abs/1506.04214
2. Ham, Y.-G., Kim, J.-H., & Luo, J.-J. (2019). Deep learning for multi-year ENSO forecasts. *Nature*, 573, 568-572. https://doi.org/10.1038/s41586-019-1559-7
3. Chattopadhyay, A., et al. (2020). Predicting clustered weather patterns (CNN espaço-temporal). *Scientific Reports*, 10, 1317. https://doi.org/10.1038/s41598-020-57897-9
4. Taylor, J., & Feng, M. (2022). A deep learning model for forecasting global monthly mean SST anomalies. *Front. Climate*, 4, 932932. https://doi.org/10.3389/fclim.2022.932932
5. Mir, A. A., et al. (2024). ENSO dataset & comparison of deep learning models. *Earth Sci. Inform.* https://doi.org/10.1007/s12145-024-01295-6

Relação completa em `Artigos_Referências/Referências_Bibliográficas.xls`.